In [15]:
import os
from skimage import io, color
import numpy as np
from skimage.feature import graycomatrix, graycoprops
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [10]:

def wytnij_probki(folder_wej, folder_wyj, rozmiar=128):
    for kategoria in os.listdir(folder_wej):                       
        os.makedirs(f'{folder_wyj}/{kategoria}', exist_ok=True)    
        nr = 0
        for plik in os.listdir(f'{folder_wej}/{kategoria}'):       
            obraz = io.imread(f'{folder_wej}/{kategoria}/{plik}')
            for y in range(0, obraz.shape[0] - rozmiar + 1, rozmiar):       
                for x in range(0, obraz.shape[1] - rozmiar + 1, rozmiar):  
                    probka = obraz[y:y+rozmiar, x:x+rozmiar]
                    io.imsave(f'{folder_wyj}/{kategoria}/{nr}.png', probka, check_contrast=False)
                    nr += 1
        print(kategoria, nr)

wytnij_probki('tekstury', 'probki', rozmiar=128)

dywan 9
gres 25
laminat 49
tynk 25


In [11]:
cechy = ['dissimilarity', 'correlation', 'contrast', 'energy', 'homogeneity', 'ASM']
wiersze = []

for kategoria in os.listdir('probki'):
    for plik in os.listdir(f'probki/{kategoria}'):
        obraz = io.imread(f'probki/{kategoria}/{plik}')
        szary = (color.rgb2gray(obraz[:, :, :3]) * 63).astype(np.uint8)    
        glcm = graycomatrix(szary, [1, 3, 5], [0, np.pi/4, np.pi/2, 3*np.pi/4],
                            levels=64, symmetric=True, normed=True)
        wiersz = []
        for c in cechy:
            wiersz.extend(graycoprops(glcm, c).flatten())    
        wiersz.append(kategoria)                             
        wiersze.append(wiersz)

print(len(wiersze), len(wiersze[0]))

108 73


In [14]:
kolumny = []
for c in cechy:
    for d in [1,3,5]:
        for k in [0,45,90,135]:
            kolumny.append(f'{c}_d{d}_k{k}')
kolumny.append('kategoria')

df = pd.DataFrame(wiersze, columns= kolumny)
df.to_csv('cechy_tekstur.csv', index=False)
print(df.shape)
print(df['kategoria'].value_counts())

(108, 73)
kategoria
laminat    49
gres       25
tynk       25
dywan       9
Name: count, dtype: int64


In [16]:
df = pd.read_csv('cechy_tekstur.csv')
X = df.drop(columns='kategoria')
Y = df['kategoria']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size= 0.3, random_state = 0, stratify=Y)

skaler = StandardScaler()
X_train = skaler.fit_transform(X_train)
X_test = skaler.transform(X_test)

model = SVC()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

print('Zbiór treningowy:', len(X_train), ' testowy:', len(X_test))
print('Dokładność:', round(accuracy_score(Y_test, Y_pred), 3))

Zbiór treningowy: 75  testowy: 33
Dokładność: 1.0
